In [1]:
import sys
sys.path.append('../src')

import os

from dataset import load_dataset, split_dataset
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from xgboost import XGBClassifier
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

## Load Yelp dataset

In [2]:
# Load the Yelp dataset and split it into train, test
labels, features, homogenous = load_dataset("../data/YelpChi.mat", "../data/yelp_home_adjlists.pickle")
xtrain, xtest, ytrain, ytest, _, _ = split_dataset(features, labels)

## Basic example with XGBoost

In [3]:
n_estimators = 100
max_depth = 6
learning_rate = 0.3
subsample = 1.0
colsample_bytree = 1.0
thres = 0.5

In [4]:
xgb_classifier = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
)

In [5]:
xgb_classifier.fit(xtrain, ytrain)

# Predict probabilities for the test set
ypred_proba = xgb_classifier.predict_proba(xtest)[:, 1]

# Convert probabilities to binary labels based on a threshold (0.5 used here)
ypred_binary = (ypred_proba > thres).astype(int)

# Calculate the ROC AUC score
roc_auc = roc_auc_score(ytest, ypred_proba)

# Calculate F1-score, Precision, and Recall
f1 = f1_score(ytest, ypred_binary)
precision = precision_score(ytest, ypred_binary)
recall = recall_score(ytest, ypred_binary)

# Print the scores
print(f"Model ROC AUC (XGBoost) = {100 * roc_auc:.2f}%")
print(f"Model F1-score (XGBoost) = {f1:.3f}")
print(f"Model Precision (XGBoost) = {precision:.3f}")
print(f"Model Recall (XGBoost) = {recall:.3f}")

Model ROC AUC (XGBoost) = 95.37%
Model F1-score (XGBoost) = 0.728
Model Precision (XGBoost) = 0.851
Model Recall (XGBoost) = 0.637


## Hyper-parameter tuning

In [7]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

# Define the search space
space = {
    "n_estimators": hp.choice("n_estimators", [50, 100, 200, 300]),
    "max_depth": hp.choice("max_depth", [3, 4, 5, 6, 7, 8]),
    "learning_rate": hp.uniform("learning_rate", 0.01, 0.5),
    "subsample": hp.uniform("subsample", 0.6, 1.0),
    "colsample_bytree": hp.uniform("colsample_bytree", 0.6, 1.0),
}

/home/trung.mai@corporate.eaglys.com/mlflow_basic/.venv/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [8]:
# Objective function: hyperopt minimizes, so we negate ROC AUC
def objective(params):
    clf = XGBClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        subsample=params["subsample"],
        colsample_bytree=params["colsample_bytree"],
        eval_metric="logloss",
        verbosity=0,
    )
    clf.fit(xtrain, ytrain)
    ypred_proba = clf.predict_proba(xtest)[:, 1]
    roc_auc = roc_auc_score(ytest, ypred_proba)
    return {"loss": -roc_auc, "status": STATUS_OK}

In [9]:
# Run hyperopt — TPE algorithm, 50 evaluations
trials = Trials()
best = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=50,
    trials=trials,
)

print("Best hyperparameters found:")
print(best)

100%|████████████████████████████████████████████████████████████████| 50/50 [00:13<00:00,  3.71trial/s, best loss: -0.9605645056713732]
Best hyperparameters found:
{'colsample_bytree': np.float64(0.8536006487468758), 'learning_rate': np.float64(0.13801040651249544), 'max_depth': np.int64(4), 'n_estimators': np.int64(3), 'subsample': np.float64(0.9766555634181473)}


In [10]:
# hp.choice returns the index, so map back to actual values
n_estimators_choices = [50, 100, 200, 300]
max_depth_choices = [3, 4, 5, 6, 7, 8]

best_clf = XGBClassifier(
    n_estimators=n_estimators_choices[best["n_estimators"]],
    max_depth=max_depth_choices[best["max_depth"]],
    learning_rate=best["learning_rate"],
    subsample=best["subsample"],
    colsample_bytree=best["colsample_bytree"],
    eval_metric="logloss",
)
best_clf.fit(xtrain, ytrain)

ypred_proba = best_clf.predict_proba(xtest)[:, 1]
ypred_binary = (ypred_proba > 0.5).astype(int)

print(f"Best Model ROC AUC  = {100 * roc_auc_score(ytest, ypred_proba):.2f}%")
print(f"Best Model F1-score = {f1_score(ytest, ypred_binary):.3f}")
print(f"Best Model Precision = {precision_score(ytest, ypred_binary):.3f}")
print(f"Best Model Recall   = {recall_score(ytest, ypred_binary):.3f}")

Best Model ROC AUC  = 96.06%
Best Model F1-score = 0.752
Best Model Precision = 0.885
Best Model Recall   = 0.655
